### Inputs:
- `input_file`: Path to the input file containing CDR data.

### Outputs:
- `CDR_over_all_plot_dic`: Dictionary to store CDR data for each chromosome. Each region is stored in a format of `[chromosome name, region start, region end, region status]`.
  - Example: 
    ```python
    {
        'chr1_PATERNAL': [['chr1', 1000, 2000, 'CDR'], ['chr1', 3000, 4000, 'CDR_Transition']],
        'chr1_MATERNAL': [['chr1', 5000, 6000, 'CDR']]
    }
    ```
- `CDR_dict`: Dictionary to store CDR region start and end for each chromosome. Each region is stored in a format of `chromosome name: [[start, end]]`.
  - Example: 
    ```python
    {
        'chr1': [[1000, 2000], [5000, 6000]]
    }
    ```
- `CDR_transition_dict`: Dictionary to store CDR transition regions. Each region is stored in a format of `chromosome name: [[start, end]]`.
  - Example: 
    ```python
    {
        'chr1': [[3000, 4000]]
    }
    ```

### Description:
This cell initializes dictionaries to store CDR data and reads the input file to populate these dictionaries. It processes each line of the input file to extract chromosome number, CDR start and end positions, and CDR status, and stores this information in the appropriate dictionaries.


In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import pysam
import numpy as np
from Bio import SeqIO
import time 
import matplotlib.patches as patches
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import itertools
import csv
import random
import matplotlib.colors as mcolors
import os

old_cdr_file = '/private/groups/migalab/dan/data_analysis/young_old_analysis/HG002_DiMeLo_CENPA_oldpassage.hmmCDR_only_CDR_Dan_certified.bed'
young_cdr_file = '/private/groups/migalab/dan/data_analysis/young_old_analysis/HG002_DiMeLo_CENPA_youngpassage.hmmCDR_only_CDR_Dan_certified.bed'
ipsc_cdr_file = '/private/groups/migalab/dan/data_analysis/hiPSC_CDR_included_domains.bed'
def process_cdr_bed_file(input_file):
    """
    Process a BED file containing CDR data and organize it into dictionaries.
    
    Args:
        input_file (str): Path to the input BED file
        
        
    Returns:
        tuple: (CDR_over_all_plot_dic, CDR_dict, CDR_transition_dict)
    """
    # Initialize dictionaries
    CDR_over_all_plot_dic = {}
    for num in range(1, 23):
        CDR_over_all_plot_dic[f'chr{num}_PATERNAL'] = []
        CDR_over_all_plot_dic[f'chr{num}_MATERNAL'] = []
    CDR_over_all_plot_dic['chrX_MATERNAL'] = []
    CDR_over_all_plot_dic['chrY_PATERNAL'] = []
    
    CDR_dict = {}
    CDR_transition_dict = {}
    
    # Process the BED file
    with open(input_file, 'r') as infile:
        for line in infile:
            # Split the line into components
            fields = line.strip().split('\t')
            chr_num = fields[0]
            CDR_start = int(fields[1])
            CDR_end = int(fields[2])
            try: 
                CDR_status = fields[3]
                
                # Store in overall plot dictionary
                CDR_over_all_plot_dic[chr_num].append([chr_num, CDR_start, CDR_end, CDR_status])
                
                # Store in specific dictionaries based on CDR status
                if CDR_status == 'CDR' or CDR_status.startswith('Domain_'):
                    if chr_num not in CDR_dict:
                        CDR_dict[chr_num] = [[CDR_start, CDR_end]]
                    else:
                        CDR_dict[chr_num].append([CDR_start, CDR_end])
                elif CDR_status == 'CDR_Transition':
                    if chr_num not in CDR_transition_dict:
                        CDR_transition_dict[chr_num] = [[CDR_start, CDR_end]]
                    else:
                        CDR_transition_dict[chr_num].append([CDR_start, CDR_end])
            except IndexError: 
                CDR_over_all_plot_dic[chr_num].append([chr_num, CDR_start, CDR_end])
                CDR_dict[chr_num] = [[CDR_start, CDR_end]]
            
    
    return CDR_dict

young_CDR_dict = process_cdr_bed_file(young_cdr_file)
old_CDR_dict = process_cdr_bed_file(old_cdr_file)
ipsc_CDR_dict = process_cdr_bed_file(ipsc_cdr_file)


### Inputs:
- `CDR_dict`: Dictionary containing CDR regions.

### Outputs:
- `varibility_data_frame`: DataFrame to store variability data for each chromosome.
  - Example:
    ```python
    hap CDR variability  CDR num difference
    chr1                NaN                NaN
    chr2                NaN                NaN
    ...
    ```
- `chromosome_CDR`: DataFrame to store total CDR length and number for each chromosome.
  - Example:
    ```python
    total CDR length  CDR num
    chr1_PATERNAL              10000        5
    chr1_MATERNAL              8000         4
    ...
    ```
- `maternal_avg`: Average total CDR length for maternal chromosomes.
  - Example:
    ```python
    8000
    ```
- `paternal_avg`: Average total CDR length for paternal chromosomes.
  - Example:
    ```python
    10000
    ```
- `overall_avg`: Overall average total CDR length.
  - Example:
    ```python
    9000
    ```

### Description:
This cell creates DataFrames to store variability data and CDR data for each chromosome. It calculates the total CDR length and number for each chromosome and stores this information in the `chromosome_CDR` DataFrame. It also calculates the average total CDR length for maternal, paternal, and overall chromosomes.

In [ ]:
import pandas as pd

def calculate_cdr_metrics(CDR_dict):
    # Initialize chromosome lists
    chromosomes = [f'chr{i}' for i in range(1, 23)]
    chromosomes_PAT = [f'chr{i}_PATERNAL' for i in range(1, 23)]
    chromosomes_MAT = [f'chr{i}_MATERNAL' for i in range(1, 23)]
    chromosomes_MAT_PAT = chromosomes_PAT + chromosomes_MAT

    # Initialize variability DataFrame
    varibility_columns = ['hap CDR variability', 'CDR num difference']
    varibility_data_frame = pd.DataFrame(index=chromosomes, columns=varibility_columns)

    # Initialize CDR DataFrame
    CDR_columns = ['total CDR length', 'CDR num']
    chromosome_CDR = pd.DataFrame(index=chromosomes_MAT_PAT, columns=CDR_columns)

    # Calculate CDR metrics
    for chromosome in CDR_dict:
        CDRs = CDR_dict[chromosome]
        total_CDR_length = 0
        total_CDR_num = 0
        for CDR in CDRs: 
            CDR_start = CDR[0]
            CDR_end = CDR[1]
            total_CDR_length += CDR_end - CDR_start
            total_CDR_num += 1 
        chromosome_CDR.at[chromosome, 'total CDR length'] = total_CDR_length
        chromosome_CDR.at[chromosome, 'CDR num'] = total_CDR_num

    # Sort and calculate averages
    chromosome_CDR = chromosome_CDR.sort_index()
    maternal_avg = chromosome_CDR[chromosome_CDR.index.str.endswith('_MATERNAL')]['total CDR length'].mean()
    paternal_avg = chromosome_CDR[chromosome_CDR.index.str.endswith('_PATERNAL')]['total CDR length'].mean()
    overall_avg = chromosome_CDR['total CDR length'].mean()

    # Create regions DataFrame
    data = []
    for chromosome, regions in CDR_dict.items():
        for region in regions:
            start, end = region
            difference = end - start
            data.append({'Chromosome': chromosome, 'Difference': difference})
            
    
    cdr_regions_df = pd.DataFrame(data)
    
    return cdr_regions_df

old_cdr_regions_df = calculate_cdr_metrics(old_CDR_dict)
young_cdr_regions_df = calculate_cdr_metrics(young_CDR_dict)
ipsc_cdr_regions_df = calculate_cdr_metrics(ipsc_CDR_dict)

In [ ]:

output_dir = Path('/private/groups/migalab/dan/data_analysis/coverage_analysis')
CENPA_AS_bam_file = '/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1_5mA_6mC_winnowmap_MD.bam'
CENPC_AS_bam_file = '/private/groups/migalab/dan/01_09_24_R1041_DiMeLoAdaptive_CENPC/01_09_24_R1041_DiMeLoAdaptive_CENPC/01_09_24_R1041_DiMeLoAdaptive_CENPC/20240109_1200_6B_PAS52674_0adbae11/pod5_pass/explicit/01_09_24_R1041_DiMeLoAdaptive_CENPC_5mC_6mA_winnowmap_sorted_MD.bam'
H3K9me3_AS_bam_file = '/private/groups/migalab/dan/08_05_24_R1041_ULadapt_Dimelo_H3K9ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_1/20240805_1148_1F_PAU87705_0451cc00/pod5/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_mA_mC_winnowmap_sorted_MD.bam'
ref_genome_file = Path('/private/groups/migalab/dan/reference/hg002v1.0.1.fasta')



np.set_printoptions(threshold=np.inf)
min_quality_score = 8

#Load the bam file 
cenpa_young_bamfile = pysam.AlignmentFile( "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1_5mA_6mC_winnowmap_MD.bam",
    "rb") 


h3k9me3_young_bamfile = pysam.AlignmentFile(
    "/private/groups/migalab/dan/08_05_24_R1041_ULadapt_Dimelo_H3K9ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_1/20240805_1148_1F_PAU87705_0451cc00/pod5/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_mA_mC_winnowmap_sorted_MD.bam",
    "rb") 

CENPC_bamfile = pysam.AlignmentFile(
    CENPC_AS_bam_file,
    'rb')

cenpa_old_bamfile = pysam.AlignmentFile(
    "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAold_1/20240611_1126_1G_PAW33047_5b119baf/pod5/06_11_24_R1041_UL_DiMeLo_CENPAold_1_5mC_6mA_winnowmap_sorted_sorted_clean_MD.bam",
    "rb") 

h3k9me3_old_bamfile = pysam.AlignmentFile(
    "/private/groups/migalab/dan/09_17_24_R1041_UL_Dimelo_H3K4me3-old/09_17_24_R1041_UL_Dimelo_H3K4me3-old_1/20240917_1924_4C_PAY89134_71d52a87/pod5/09_17_24_R1041_UL_Dimelo_H3K4me3_old_winnowmap_sorted_sorted_clean_MD.bam",
    "rb") 

ipsc_cenpa_bamfile = pysam.AlignmentFile("/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/merged_UL_DiMeLo_CENPAyoung_1_5mA_6mC_winnowmap_MD_corrected.bam",
    "rb") 




In [ ]:
'''
The idea of this function is to isolate the the desired regions (here in the function, it is called the subset) in the mod 
numpy array without dashes(insertions)'''

def mod_subset_producing_step (mod_no_dash,alignment_dash,target_start_no_dash,target_end_no_dash):
    #mod_no_dash = is the numpy array of the mod without any insertions and deletions
    # alignment_dash = is the alignment sequence with the dashes in it 
    # target_start = it's the subset starting position WITHOUT the dashes!!! 


    # Create a mask to identify non-dash positions
    mask = [char != '-' for char in alignment_dash]

    # Generate cumulative counts only for True values in the mask
    cumulative_counts = list(itertools.accumulate(mask))


    
    # Create the final indexes list
    indexes = [count - 1 if is_non_dash else '-' for count, is_non_dash in zip(cumulative_counts, mask)]



    target_start_dash = indexes.index (target_start_no_dash)

        
    try:
        target_end_dash = indexes.index (target_end_no_dash)
    except ValueError: 
        target_end_dash = indexes[-1]




    #obtain dashed alignment 
    alignment_dash_sequence_pre_subset = alignment_dash[0:target_start_dash]
    alignment_dash_sequence_subset = alignment_dash[target_start_dash:target_end_dash]

    #create no dash alignment 
    alignment_no_dash_sequence_pre_subset = alignment_dash_sequence_pre_subset.replace("-","")
    alignment_no_dash_sequence_subset = alignment_dash_sequence_subset.replace("-","")

    subset_no_dash_start = len(alignment_no_dash_sequence_pre_subset)
    subset_no_dash_end = subset_no_dash_start + len(alignment_no_dash_sequence_subset)

    #make mod_no_dash alignment
    mod_subset = mod_no_dash[subset_no_dash_start:subset_no_dash_end]

    return mod_subset





In [ ]:

def region_read_mA_density_calculator (chr_name,region_start_index,region_end_index,bamfile,mod_tag,filtering_val): 
    data_table = [] 
    region_density = []
    region_base = 0 
            


    for read in bamfile.fetch(chr_name,region_start_index,region_end_index):
        #make an if statement to check a specific read front, middle, end regions 
        #setting read start, end, density, length variables 
            
        #Get the starting and ending positions of the reads 
        read_start_position = read.reference_start
        read_end_position = read.reference_end
        read_density = 0 

        #Get sequence information which shows deletions and insertions 
        sequence = read.get_aligned_pairs(matches_only=False, with_seq = True)


        #make a numpy of the sequence length which eliminates the deletion
        
        read_sequence_insertion_included = ''
        genomic_alignment_sequence_deletion_mistach_included = ''
        
        for item in sequence:
            if item[0] is None:
                read_sequence_insertion_included+='-'
            elif item[1] is None:
                genomic_alignment_sequence_deletion_mistach_included += '-'
            else: 
                read_sequence_insertion_included+=item[2]
                genomic_alignment_sequence_deletion_mistach_included +=item[2]

        
        read_sequence_insertion_included = read_sequence_insertion_included.upper()
        genomic_alignment_sequence_deletion_mistach_included = genomic_alignment_sequence_deletion_mistach_included.upper()


        genomic_alignment_sequence_deletion_mistach_included_mask = np.array(
            [char != '-' for char in genomic_alignment_sequence_deletion_mistach_included])

        #take sequence length excluding insertions 
        insertions = read_sequence_insertion_included.count ("-")
        no_insertion_no_deletion_sequence_length = len(read_sequence_insertion_included) 
        
        
        mod=read.modified_bases_forward
        
        #make a mod score with its original length 
        mod_score = np.zeros(len(genomic_alignment_sequence_deletion_mistach_included),)

        
        #make transfer mA positions to mod np array corresponded to their sequence positions 
        try:
            if mod_tag == 'A':
                for indices, values in mod[('A', 0, 'a')]:
                    mod_score[indices] = values
                mod_score = mod_score[genomic_alignment_sequence_deletion_mistach_included_mask]
                


            elif mod_tag == 'CG':
                for indices, values in mod[('C', 0, 'm')]:
                    mod_score[indices] = values
                mod_score = mod_score[genomic_alignment_sequence_deletion_mistach_included_mask]
            
            if read.is_reverse:
                    mod_score = mod_score[::-1]
            


        # No mod would return KeyError 
        except KeyError:
            continue
        mod_score = np.where(mod_score < filtering_val, 0, mod_score)
        # if the regions are longer than the reads 
        if (region_end_index - region_start_index) > (read_end_position - read_start_position):
            # scenario 4: if the reads are inside the region
            if (region_end_index >= read_end_position) and (region_start_index <= read_start_position): 
                mod_start = 0
                mod_end = len(read_sequence_insertion_included)
            
            # scenario 5: if the reads cover the later part of the region
            elif (region_end_index < read_end_position) and (region_start_index > read_start_position): 
                mod_start = 0
                mod_end = no_insertion_no_deletion_sequence_length - read_end_position - region_end_index

            # scenario 6: if the reads cover the starting part of the region 
            elif (region_end_index > read_end_position) and (region_start_index > read_start_position): 
                mod_start = region_start_index - read_start_position 
                mod_end = no_insertion_no_deletion_sequence_length

                
        
        # if the reads are longer than the region selected 
        elif (region_end_index - region_start_index) <= (read_end_position - read_start_position):
            # scenario 1: when the defined region is inside the read
            if (read_start_position <= region_start_index) and (read_end_position >= region_end_index):
                mod_start = region_start_index - read_start_position 
                mod_end = region_end_index - read_start_position

            # scenario 3: when the defined region covers a bit of the end of the read
            elif (read_end_position < region_end_index) and (read_end_position > region_start_index):
                mod_start = region_start_index - read_start_position

                mod_end = no_insertion_no_deletion_sequence_length

            # scenario 2: when the defined region covers a bit of the beginning of the read
            elif (read_start_position > region_start_index) and (read_start_position < region_end_index):
                mod_start = 0
                mod_end = region_end_index - read_start_position 


        #use the defined starting and ending positons in the region to subset mod numpy
        if (region_start_index - read_start_position) > (no_insertion_no_deletion_sequence_length - insertions):
            continue
        try:
            trimmed_mod_score = mod_subset_producing_step (mod_score,read_sequence_insertion_included,mod_start,mod_end)
        except ValueError:
            continue
        
    
        region_base += (mod_end - mod_start)
        #removing all the zeros 
        mod_no_zeros = trimmed_mod_score[trimmed_mod_score != 0]
        m_mod_tag = len (mod_no_zeros)
        

        #Getting the total amount of As in the subsetted region of the sequence 
        total_mod_tag = read_sequence_insertion_included[mod_start:mod_end].count(mod_tag)
        
        
        #calculate read density
        try:
            read_density = m_mod_tag / total_mod_tag
            
        except ZeroDivisionError:
            pass
        region_density.append (read_density)

        
                
            
        #calculate averaged region density average 
    try:
        region_density_average = sum(region_density)/len(region_density)
    
    except ZeroDivisionError:
        region_density_average = 0
    return region_density_average



In [ ]:

def match_cdr_regions(CDR_dict, cdr_regions_df, bamfile):
    """
    Matches CDR regions by size and chromosome from two different datasets 
    and calculates mA and mC densities for the matched regions.

    Args:
        CDR_dict (dict): Dictionary containing chromosomes as keys and lists of CDR start-end tuples as values.
        cdr_regions_df (pd.DataFrame): DataFrame containing the chromosome and region size information.
        bamfile (str): Path to the BAM file used for density calculations.

    Returns:
        pd.DataFrame: DataFrame containing matched CDR regions with calculated mA and mC densities.
    """
    # Step 1: Define columns for the DataFrame
    CDR_columns_individual = ['line number', 'chromosome', 'CDR_start', 'CDR_end']

    # Step 2: Initialize variables
    rows = []
    n = 0

    # Step 3: Populate the chromosome_individual_CDR DataFrame
    for chromosome, CDRs in CDR_dict.items():
        for CDR in CDRs:
            n += 1
            CDR_start, CDR_end = CDR
            rows.append({
                'line number': n,
                'chromosome': chromosome,
                'CDR_start': CDR_start,
                'CDR_end': CDR_end
            })

    # Convert rows to DataFrame
    chromosome_individual_CDR = pd.DataFrame(rows, columns=CDR_columns_individual)

    # Step 4: Calculate the CDR size
    chromosome_individual_CDR['Size'] = chromosome_individual_CDR['CDR_end'] - chromosome_individual_CDR['CDR_start']

    # Step 5: Match CDRs by size and chromosome
    matched_data = []

    for index, region_row in cdr_regions_df.iterrows():

        chromosome = region_row['Chromosome']
        region_size = region_row['Difference']

        match = chromosome_individual_CDR[
            (chromosome_individual_CDR['chromosome'] == chromosome) & 
            (chromosome_individual_CDR['Size'] == region_size)
        ]

        if not match.empty:
            matched_data.append({
                'Chromosome': chromosome,
                'Region Length': region_size,
                'CDR_start': match['CDR_start'].values[0],
                'CDR_end': match['CDR_end'].values[0],
                'mA density': region_read_mA_density_calculator(chromosome, match['CDR_start'].values[0], match['CDR_end'].values[0], bamfile, "A", 240),
                'mC density': region_read_mA_density_calculator(chromosome, match['CDR_start'].values[0], match['CDR_end'].values[0], bamfile, "CG", 230),
            })

    # Step 6: Convert matched data to a DataFrame
    matched_cdr_df = pd.DataFrame(matched_data)

    return matched_cdr_df



In [ ]:

ipsc_CENPA_match_cdr_regions = match_cdr_regions(ipsc_CDR_dict, ipsc_cdr_regions_df, ipsc_cenpa_bamfile)

print (ipsc_CENPA_match_cdr_regions)

In [ ]:
young_CENPA_match_cdr_regions = match_cdr_regions(young_CDR_dict, young_cdr_regions_df, cenpa_young_bamfile)


In [ ]:

young_CENPA_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/young_CENPA_match_cdr_regions.csv', index=False)

In [ ]:
old_CENPA_match_cdr_regions = match_cdr_regions(old_CDR_dict, old_cdr_regions_df, cenpa_old_bamfile)

In [ ]:
old_CENPA_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/old_CENPA_match_cdr_regions.csv', index=False)

In [ ]:
ipsc_CENPA_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/ipsc_cenpa_match_cdr_regions.csv', index=False)

In [ ]:
young_h3k9_match_cdr_regions = match_cdr_regions(young_CDR_dict, young_cdr_regions_df, h3k9me3_young_bamfile)
old_h3k9_match_cdr_regions = match_cdr_regions(old_CDR_dict, old_cdr_regions_df, h3k9me3_old_bamfile)


In [ ]:
young_h3k9_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/young_h3k9_match_cdr_regions.csv', index=False)
old_h3k9_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/old_h3k9_match_cdr_regions.csv', index=False)

In [ ]:
CENPC_match_cdr_regions = match_cdr_regions(young_CDR_dict, young_cdr_regions_df, CENPC_bamfile)

In [ ]:
CENPC_match_cdr_regions.to_csv('/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/CENPC_match_cdr_regions.csv', index=False)

In [ ]:
young_h3k9_match_cdr_mC_regions = match_cdr_regions(old_CDR_dict, old_cdr_regions_df, h3k9me3_young_bamfile)

In [ ]:


def load_young_CENPA_match_cdr_regions(csv_path: str) -> pd.DataFrame:
    # Read CSV, drop any accidental index columns
    df = pd.read_csv(csv_path)
    df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed')]

    # Canonical column names we expect from your writer
    canonical = ["Chromosome", "Region Length", "CDR_start", "CDR_end", "mA density", "mC density"]

    # If someone lowercased headers or changed spacing, normalize them back
    # (works even if they’re already correct)
    lower_to_actual = {c.lower(): c for c in df.columns.astype(str)}
    alias_map = {
        "chromosome": "Chromosome",
        "chr": "Chromosome",
        "region length": "Region Length",
        "length": "Region Length",
        "cdr_start": "CDR_start",
        "cdr end": "CDR_end",
        "cdr_end": "CDR_end",
        "ma density": "mA density",
        "maden": "mA density",
        "mc density": "mC density",
        "mcdensity": "mC density",
    }
    for k_lower, target in alias_map.items():
        if k_lower in lower_to_actual and target not in df.columns:
            df.rename(columns={lower_to_actual[k_lower]: target}, inplace=True)

    # Ensure required columns exist
    missing = [c for c in canonical if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing expected columns: {missing}")

    # Enforce dtypes
    df["Chromosome"] = df["Chromosome"].astype(str)
    for col in ["Region Length", "CDR_start", "CDR_end"]:
        df[col] = pd.to_numeric(df[col], errors="raise").astype("int64")
    for col in ["mA density", "mC density"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")

    # Optional: clip densities into [0,1] if any stray values exist
    df["mA density"] = df["mA density"].clip(lower=0, upper=1)
    df["mC density"] = df["mC density"].clip(lower=0, upper=1)

    # Sanity check (and fix) Region Length
    computed_len = df["CDR_end"] - df["CDR_start"]
    if not (computed_len == df["Region Length"]).all():
        # If there’s a mismatch, trust the coordinates and recompute length
        df["Region Length"] = computed_len

    # Sort for convenience
    df = df.sort_values(["Chromosome", "CDR_start", "CDR_end"]).reset_index(drop=True)
    return df

# ---- Use it ----
young_CENPA_csv_path = '/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/young_CENPA_match_cdr_regions.csv'
old_CENPA_csv_path = "/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/old_CENPA_match_cdr_regions.csv"
young_h3k9_csv_path = "/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/young_h3k9_match_cdr_regions.csv"
old_h3k9_csv_path = "/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/old_h3k9_match_cdr_regions.csv"
cenpc_csv_path ='/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/CENPC_match_cdr_regions.csv'
ipsc_cenpa_path = '/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot/ipsc_cenpa_match_cdr_regions.csv'
young_CENPA_match_cdr_regions = load_young_CENPA_match_cdr_regions(young_CENPA_csv_path)
old_CENPA_match_cdr_regions = load_young_CENPA_match_cdr_regions(old_CENPA_csv_path)
young_h3k9_match_cdr_regions = load_young_CENPA_match_cdr_regions(young_h3k9_csv_path)
old_h3k9_match_cdr_regions = load_young_CENPA_match_cdr_regions(old_h3k9_csv_path)
cenpc_match_cdr_regions = load_young_CENPA_match_cdr_regions (cenpc_csv_path)
ipsc_CENPA_match_cdr_regions = load_young_CENPA_match_cdr_regions(ipsc_cenpa_path)


### Inputs:
- `cdr_regions_df`: DataFrame containing CDR regions with chromosome and difference (length).
- `chromosome_CDR`: DataFrame containing total CDR length and number for each chromosome.
- `maternal_avg`: Average total CDR length for maternal chromosomes.
- `paternal_avg`: Average total CDR length for paternal chromosomes.
- `overall_avg`: Overall average total CDR length.

### Outputs:
- A plot showing total CDR lengths in chromosomes for young passage.

### Description:
This cell creates a plot to visualize the total CDR lengths in chromosomes for young passage. It iterates through each chromosome and plots the regions, with different colors for maternal and paternal data. It also adds horizontal lines for the average CDR lengths.

In [ ]:

def fill_cdr_gaps(matched_cdr_df_sorted):
    """
    Identifies gaps between consecutive CDR regions within each chromosome 
    and fills those gaps with new rows in the DataFrame.
    
    Parameters:
        matched_cdr_df_sorted (pd.DataFrame): A sorted DataFrame containing CDR regions.
        
    Returns:
        pd.DataFrame: A DataFrame with gaps filled between CDR regions.
    """
    gap_rows = []

    for chromosome, group in matched_cdr_df_sorted.groupby('Chromosome'):
        group_sorted = group.sort_values('CDR_start').reset_index(drop=True)
        for i in range(1, len(group_sorted)):
            prev_cdr_end = group_sorted.at[i-1, 'CDR_end']
            current_cdr_start = group_sorted.at[i, 'CDR_start']
            gap_length = current_cdr_start - prev_cdr_end
            if gap_length > 0:
                gap_row = {
                    'Chromosome': chromosome,
                    'Region Length': gap_length,
                    'CDR_start': prev_cdr_end,
                    'CDR_end': current_cdr_start,
                    'mA density': 0.0,
                    'mC density': 100.0
                }
                gap_rows.append(gap_row)

    gaps_df = pd.DataFrame(gap_rows, columns=matched_cdr_df_sorted.columns)
    matched_cdr_df_with_gaps = pd.concat([matched_cdr_df_sorted, gaps_df], ignore_index=True)
    matched_cdr_df_with_gaps = matched_cdr_df_with_gaps.sort_values(['Chromosome', 'CDR_start']).reset_index(drop=True)

    return matched_cdr_df_with_gaps

young_CENPA_matched_cdr_df_with_gaps = fill_cdr_gaps(young_CENPA_match_cdr_regions)
old_CENPA_matched_cdr_df_with_gaps = fill_cdr_gaps(old_CENPA_match_cdr_regions)

young_h3k9_match_cdr_regions_with_gaps = fill_cdr_gaps(young_h3k9_match_cdr_regions)
old_h3k9_match_cdr_regions_with_gaps = fill_cdr_gaps(old_h3k9_match_cdr_regions)
ipsc_match_cdr_regions_with_gaps = fill_cdr_gaps(ipsc_CENPA_match_cdr_regions)
CENPC_matched_cdr_df_with_gaps = fill_cdr_gaps(cenpc_match_cdr_regions)




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats  # add at top of script

def _scatter_with_identity(ax, x, y, xlabel, ylabel, title,
                           xlim=(0, 0.05), ylim=(0, 0.4),
                           add_fit=True):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # scatter
    ax.plot(x, y, marker='o', linestyle='None', alpha=0.7)

    # fixed axes
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)

    # square panel
    try:
        ax.set_box_aspect(1)
    except AttributeError:
        pass

    if add_fit:
        mask = (
            np.isfinite(x) & np.isfinite(y) &
            (x >= xlim[0]) & (x <= xlim[1]) &
            (y >= ylim[0]) & (y <= ylim[1])
        )
        if mask.sum() >= 2:
            x_fit = x[mask]
            y_fit = y[mask]

            if np.ptp(x_fit) < 1e-12:
                y_bar = float(np.mean(y_fit))
                ax.plot([xlim[0], xlim[1]], [y_bar, y_bar])
                fit_lbl = f"fit: x≈const, ȳ={y_bar:.3g}"
                r2, pval = np.nan, np.nan
            else:
                m, b = np.polyfit(x_fit, y_fit, deg=1)
                xx = np.linspace(xlim[0], xlim[1], 200)
                yy = m * xx + b
                ax.plot(xx, yy)

                # correlation + p-value
                r, pval = stats.pearsonr(x_fit, y_fit)
                y_hat = m * x_fit + b
                ss_res = np.sum((y_fit - y_hat) ** 2)
                ss_tot = np.sum((y_fit - np.mean(y_fit)) ** 2)
                r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

                fit_lbl = f"fit: y={m:.3g}x+{b:.3g}"

            # annotate with both R² and p
            ax.text(0.02, 0.98,
                    f"{fit_lbl}\nR²={r2:.3f}, p={pval:.3g}",
                    transform=ax.transAxes, va='top', ha='left')

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)



# --- 1) young_CENPA: mA vs mC ---
fig1, ax1 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax1,
    young_CENPA_match_cdr_regions["mA density"],
    young_CENPA_match_cdr_regions["mC density"],
    xlabel="mA density (young CENP-A)",
    ylabel="mC density (young CENP-A)",
    title="Young CENP-A: mA vs mC"
)
fig1.tight_layout()
fig1.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/young_CENPA_mA_vs_mC.png", dpi=1600)

# --- 2) young_h3k9: mA vs mC ---
fig2, ax2 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax2,
    young_h3k9_match_cdr_regions["mA density"],
    young_h3k9_match_cdr_regions["mC density"],
    xlabel="mA density (young H3K9me3)",
    ylabel="mC density (young H3K9me3)",
    title="Young H3K9me3: mA vs mC"
)
fig2.tight_layout()
fig2.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/young_H3K9_mA_vs_mC.png", dpi=1600)

# --- 3) mA: young CENP-A vs young H3K9me3 ---
# merge on exact region identity (Chromosome + coordinates). Adjust if your keys differ.
key_cols = ["Chromosome", "CDR_start", "CDR_end"]
cenpa_mA = young_CENPA_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_CENPA"})
h3k9_mA  = young_h3k9_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_H3K9"})
merged = pd.merge(cenpa_mA, h3k9_mA, on=key_cols, how="inner").dropna(subset=["mA_CENPA", "mA_H3K9"])

fig3, ax3 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax3,
    merged["mA_CENPA"],
    merged["mA_H3K9"],
    xlabel="mA density (young CENP-A)",
    ylabel="mA density (young H3K9me3)",
    title="mA: young CENP-A vs young H3K9me3",
    xlim=(0, 0.05),   # <-- only for the last plot
    ylim=(0, 0.05)    # <-- only for the last plot
)
fig3.tight_layout()
fig3.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/young_CENPA_vs_H3K9_mA.png", dpi=1600)

# --- A) cenpc: mA vs mC ---
fig4, ax4 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax4,
    cenpc_match_cdr_regions["mA density"],
    cenpc_match_cdr_regions["mC density"],
    xlabel="mA density (CENP-C)",
    ylabel="mC density (CENP-C)",
    title="CENP-C: mA vs mC"  # xlim=(0,0.05), ylim=(0,0.4) by helper default
)
fig4.tight_layout()
fig4.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/cenpc_mA_vs_mC.png", dpi=1600)

# --- B) mA: cenpc vs young H3K9me3 ---
key_cols = ["Chromosome", "CDR_start", "CDR_end"]
cenpc_mA = cenpc_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_CENPC"})
h3k9_mA  = young_h3k9_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_H3K9"})
merged_cenpc_h3k9 = pd.merge(cenpc_mA, h3k9_mA, on=key_cols, how="inner").dropna(subset=["mA_CENPC", "mA_H3K9"])

fig5, ax5 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax5,
    merged_cenpc_h3k9["mA_CENPC"],
    merged_cenpc_h3k9["mA_H3K9"],
    xlabel="mA density (CENP-C)",
    ylabel="mA density (young H3K9me3)",
    title="mA: CENP-C vs young H3K9me3",
    xlim=(0, 0.05),  # same tight mA range as your other cross-file mA plot
    ylim=(0, 0.05)
)
fig5.tight_layout()
fig5.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/cenpc_vs_H3K9_mA.png", dpi=1600)



plt.show()


In [ ]:
# --- C) mA: young CENP-A vs CENP-C ---
key_cols = ["Chromosome", "CDR_start", "CDR_end"]
cenpa_mA = young_CENPA_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_CENPA"})
cenpc_mA = cenpc_match_cdr_regions[key_cols + ["mA density"]].rename(columns={"mA density": "mA_CENPC"})
merged_cenpa_cenpc = pd.merge(cenpa_mA, cenpc_mA, on=key_cols, how="inner").dropna(subset=["mA_CENPA", "mA_CENPC"])

fig6, ax6 = plt.subplots(figsize=(5,5))
_scatter_with_identity(
    ax6,
    merged_cenpa_cenpc["mA_CENPA"],
    merged_cenpa_cenpc["mA_CENPC"],
    xlabel="mA density (young CENP-A)",
    ylabel="mA density (CENP-C)",
    title="mA: young CENP-A vs CENP-C",
    xlim=(0, 0.05),
    ylim=(0, 0.05)
)
fig6.tight_layout()
fig6.savefig("/private/groups/migalab/dan/data_analysis/coverage_analysis/young_CENPA_vs_CENPC_mA.png", dpi=1600)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
import os
import numpy as np

def plot_cdr_lengths_with_density_no_gap(cdr_df_with_gaps, name, output_dir, dpi=1600, lower_threshold=None, upper_threshold=0.005):
    """
    Plots total CDR lengths in chromosomes with mA density mapping, highlights maternal and paternal data,
    adds a dashed line for the average region length per homolog (maternal and paternal kept separate),
    and saves the output as a PNG and SVG file.

    Parameters:
        cdr_df_with_gaps (pd.DataFrame): DataFrame containing CDR regions and mA density.
        name (str): Name prefix for the output file.
        output_dir (str): Directory where the output PNG file should be saved.
        dpi (int, optional): Resolution of the output image in dots per inch. Default is 1600.
        lower_threshold (float, optional): Lower bound threshold for the color gradient. If None, determined from data.
        upper_threshold (float, optional): Upper bound threshold for the color gradient. Default is 0.035.
    """
    colormap = plt.cm.plasma

    # Calculate non-zero mA density from the DataFrame
    non_zero_densities = cdr_df_with_gaps[cdr_df_with_gaps['mA density'] > 0]['mA density']
    computed_min_density = non_zero_densities.min()
    computed_max_density = non_zero_densities.max()

    # Use provided thresholds if available; otherwise, use computed values
    min_density = lower_threshold if lower_threshold is not None else computed_min_density
    max_density = upper_threshold if upper_threshold is not None else computed_max_density

    # Sort chromosomes in the desired order (pair maternal and paternal together)
    chromosome_order = [item for i in range(1, 23) for item in (f'chr{i}_MATERNAL', f'chr{i}_PATERNAL')] + \
                       ["chrX_MATERNAL", "chrX_PATERNAL", "chrY_MATERNAL", "chrY_PATERNAL"]
    cdr_df_with_gaps['Chromosome'] = pd.Categorical(
        cdr_df_with_gaps['Chromosome'], categories=chromosome_order, ordered=True
    )

    cdr_df_with_gaps = cdr_df_with_gaps.sort_values('Chromosome', kind='mergesort')

    # Create a homolog identifier (e.g., '1m' for chr1_MATERNAL, '1p' for chr1_PATERNAL)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Chromosome'].str.replace(r'chr(\w+)_MATERNAL', r'\1m', regex=True)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Homolog'].str.replace(r'chr(\w+)_PATERNAL', r'\1p', regex=True)

    # Sum region lengths for each homolog separately
    total_lengths_per_homolog = cdr_df_with_gaps.groupby('Homolog')['Region Length'].sum()
    # Compute the average region length across all homologs
    average_length = total_lengths_per_homolog.mean()

    plt.figure(figsize=(15, 7))

    for chromosome in cdr_df_with_gaps['Chromosome'].unique():
        regions = cdr_df_with_gaps[cdr_df_with_gaps['Chromosome'] == chromosome]
        total_height = 0
        for _, region in regions.iterrows():
            length = region['Region Length']
            density = region['mA density']
            # Normalize the density using the selected bounds; if density is zero, color it white
            norm_val = (density - min_density) / (max_density - min_density) if density > 0 else 0
            color = colormap(norm_val) if density > 0 else 'white'
            plt.bar(chromosome, length, bottom=total_height, color=color, edgecolor='yellow')
            total_height += length

    # Add a dashed line for the average region length across the plot
    plt.axhline(y=average_length, color='black', linestyle='--')
    plt.legend()

    # Add a color bar to show the density gradient
    norm = mcolors.Normalize(vmin=min_density, vmax=max_density)
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', pad=0.01)
    cbar.set_label('mA Density', rotation=270, labelpad=15)

    plt.xticks(rotation=90)

    plt.title('Total CDR Lengths in Chromosome CDR')
    plt.xlabel('Chromosome')
    plt.ylabel('Total CDR Length')
    plt.tight_layout()

    # Save as PNG
    png_path = os.path.join(output_dir, name + "_cdr_density_plot.png")
    plt.savefig(png_path, format="png", dpi=dpi)

    # Save as SVG
    svg_path = os.path.join(output_dir, name + "_cdr_density_plot.svg")
    plt.savefig(svg_path, format="svg")

    plt.show()
    plt.close()

    print(f"PNG saved to {png_path}")
    print(f"SVG saved to {svg_path}")


def plot_cdr_lengths_with_density_no_gap_greyscale(cdr_df_with_gaps, name, output_dir, dpi=1600, lower_threshold=None, upper_threshold=0.035):
    """
    Greyscale version: Plots total CDR lengths in chromosomes with mA density mapping.
    All bars are black and the lines separating the boxes are grey.
    Saves the output as both a PNG and SVG file.

    Parameters:
        cdr_df_with_gaps (pd.DataFrame): DataFrame containing CDR regions and mA density.
        name (str): Name prefix for the output file.
        output_dir (str): Directory where the output files should be saved.
        dpi (int, optional): Resolution of the output image in dots per inch. Default is 1600.
        lower_threshold (float, optional): Lower bound threshold for the color gradient. If None, determined from data.
        upper_threshold (float, optional): Upper bound threshold for the color gradient. Default is 0.035.
    """

    # Sort chromosomes in the desired order (pair maternal and paternal together)
    chromosome_order = [item for i in range(1, 23) for item in (f'chr{i}_MATERNAL', f'chr{i}_PATERNAL')] + \
                       ["chrX_MATERNAL", "chrX_PATERNAL", "chrY_MATERNAL", "chrY_PATERNAL"]
    cdr_df_with_gaps['Chromosome'] = pd.Categorical(
        cdr_df_with_gaps['Chromosome'], categories=chromosome_order, ordered=True
    )

    cdr_df_with_gaps = cdr_df_with_gaps.sort_values('Chromosome', kind='mergesort')

    # Create a homolog identifier (e.g., '1m' for chr1_MATERNAL, '1p' for chr1_PATERNAL)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Chromosome'].str.replace(r'chr(\w+)_MATERNAL', r'\1m', regex=True)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Homolog'].str.replace(r'chr(\w+)_PATERNAL', r'\1p', regex=True)

    # Sum region lengths for each homolog separately
    total_lengths_per_homolog = cdr_df_with_gaps.groupby('Homolog')['Region Length'].sum()
    # Compute the average region length across all homologs
    average_length = total_lengths_per_homolog.mean()

    plt.figure(figsize=(15, 7))

    for chromosome in cdr_df_with_gaps['Chromosome'].unique():
        regions = cdr_df_with_gaps[cdr_df_with_gaps['Chromosome'] == chromosome]
        total_height = 0
        for _, region in regions.iterrows():
            length = region['Region Length']
            # All bars are black; separating edges are grey
            plt.bar(chromosome, length, bottom=total_height, color='black', edgecolor='grey')
            total_height += length

    # Add a dashed line for the average region length
    plt.axhline(y=average_length, color='grey', linestyle='--')
    plt.legend()

    plt.xticks(rotation=90)

    plt.title('Total CDR Lengths in Chromosome CDR')
    plt.xlabel('Chromosome')
    plt.ylabel('Total CDR Length')
    plt.tight_layout()

    # Save as PNG
    png_path = os.path.join(output_dir, name + "_cdr_density_plot_greyscale.png")
    plt.savefig(png_path, format="png", dpi=dpi)

    # Save as SVG
    svg_path = os.path.join(output_dir, name + "_cdr_density_plot_greyscale.svg")
    plt.savefig(svg_path, format="svg")

    plt.show()
    plt.close()

    print(f"PNG saved to {png_path}")
    print(f"SVG saved to {svg_path}")


# --- Run all plots (original + greyscale) ---

output_dir = "/private/groups/migalab/dan/"

# Original (colour) plots
#plot_cdr_lengths_with_density_no_gap(young_h3k9_match_cdr_regions, "young_H3K9me3", output_dir)
#plot_cdr_lengths_with_density_no_gap(old_h3k9_match_cdr_regions, "old_H3K9me3", output_dir)
#plot_cdr_lengths_with_density_no_gap(cenpc_match_cdr_regions, "CENPC", output_dir)
#plot_cdr_lengths_with_density_no_gap(young_CENPA_match_cdr_regions, "young_CENPA", output_dir)
#plot_cdr_lengths_with_density_no_gap(old_CENPA_match_cdr_regions, "old_CENPA", output_dir)
plot_cdr_lengths_with_density_no_gap(ipsc_CENPA_match_cdr_regions, "ipsc", output_dir)

# Greyscale plots
#plot_cdr_lengths_with_density_no_gap_greyscale(young_h3k9_match_cdr_regions, "young_H3K9me3_grey", output_dir)
#plot_cdr_lengths_with_density_no_gap_greyscale(old_h3k9_match_cdr_regions, "old_H3K9me3_grey", output_dir)
#plot_cdr_lengths_with_density_no_gap_greyscale(cenpc_match_cdr_regions, "CENPC_grey", output_dir)
#plot_cdr_lengths_with_density_no_gap_greyscale(young_CENPA_match_cdr_regions, "young_CENPA_grey", output_dir)
#plot_cdr_lengths_with_density_no_gap_greyscale(old_CENPA_match_cdr_regions, "old_CENPA_grey", output_dir)
#plot_cdr_lengths_with_density_no_gap_greyscale(ipsc_CENPA_match_cdr_regions, "ipsc_grey", output_dir)

In [ ]:
# Calculate the population mean mA density
mean_mA = ipsc_CENPA_match_cdr_regions["mA density"].mean()
upper_threshold = mean_mA * 2
lower_threshold = mean_mA / 2

print(f"Population mean mA density: {mean_mA:.6f}")
print(f"Upper threshold (2x mean): {upper_threshold:.6f}")
print(f"Lower threshold (0.5x mean): {lower_threshold:.6f}")

# Filter rows exceeding either threshold
outliers = ipsc_CENPA_match_cdr_regions[
    (ipsc_CENPA_match_cdr_regions["mA density"] > upper_threshold) |
    (ipsc_CENPA_match_cdr_regions["mA density"] < lower_threshold)
].copy()

outliers["fold_change"] = outliers["mA density"] / mean_mA
outliers["direction"] = outliers["fold_change"].apply(
    lambda fc: "above" if fc > 1 else "below"
)

print(f"\nRows with mA density > 2x or < 0.5x population mean ({len(outliers)} found):")
print(outliers)

In [ ]:

def plot_cdr_lengths_with_density_no_gap_mC(cdr_df_with_gaps, name, output_dir, dpi=1600, lower_threshold=None, upper_threshold=0.5):
    """
    Plots total CDR lengths in chromosomes with mA density mapping, highlights maternal and paternal data,
    adds a dashed line for the average region length per homolog (maternal and paternal kept separate),
    and saves the output as a PNG file.

    Parameters:
        cdr_df_with_gaps (pd.DataFrame): DataFrame containing CDR regions and mA density.
        name (str): Name prefix for the output file.
        output_dir (str): Directory where the output PNG file should be saved.
        dpi (int, optional): Resolution of the output image in dots per inch. Default is 1600.
        lower_threshold (float, optional): Lower bound threshold for the color gradient. If None, determined from data.
        upper_threshold (float, optional): Upper bound threshold for the color gradient. Default is 0.035.
    """
    colormap = plt.cm.plasma

    # Calculate non-zero mA density from the DataFrame
    non_zero_densities = cdr_df_with_gaps[cdr_df_with_gaps['mC density'] > 0]['mC density']
    computed_min_density = non_zero_densities.min()
    computed_max_density = non_zero_densities.max()

    # Use provided thresholds if available; otherwise, use computed values
    min_density = lower_threshold if lower_threshold is not None else computed_min_density
    max_density = upper_threshold if upper_threshold is not None else computed_max_density

    # Sort chromosomes in the desired order (pair maternal and paternal together)
    chromosome_order = [item for i in range(1, 23) for item in (f'chr{i}_MATERNAL', f'chr{i}_PATERNAL')] + \
                       ["chrX_MATERNAL", "chrX_PATERNAL", "chrY_MATERNAL", "chrY_PATERNAL"]
    cdr_df_with_gaps['Chromosome'] = pd.Categorical(
        cdr_df_with_gaps['Chromosome'], categories=chromosome_order, ordered=True
    )

    cdr_df_with_gaps = cdr_df_with_gaps.sort_values('Chromosome', kind='mergesort')

    # Create a homolog identifier (e.g., '1m' for chr1_MATERNAL, '1p' for chr1_PATERNAL)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Chromosome'].str.replace(r'chr(\w+)_MATERNAL', r'\1m', regex=True)
    cdr_df_with_gaps['Homolog'] = cdr_df_with_gaps['Homolog'].str.replace(r'chr(\w+)_PATERNAL', r'\1p', regex=True)

    # Sum region lengths for each homolog separately
    total_lengths_per_homolog = cdr_df_with_gaps.groupby('Homolog')['Region Length'].sum()
    # Compute the average region length across all homologs
    average_length = total_lengths_per_homolog.mean()



    plt.figure(figsize=(15, 7))

    for chromosome in cdr_df_with_gaps['Chromosome'].unique():
        regions = cdr_df_with_gaps[cdr_df_with_gaps['Chromosome'] == chromosome]
        total_height = 0
        for _, region in regions.iterrows():
            length = region['Region Length']
            density = region['mC density']
            # Normalize the density using the selected bounds; if density is zero, color it white
            norm_val = (density - min_density) / (max_density - min_density) if density > 0 else 0
            color = colormap(norm_val) if density > 0 else 'white'
            plt.bar(chromosome, length, bottom=total_height, color=color, edgecolor='yellow')
            total_height += length

    # Add a dashed line for the average region length across the plot
    plt.axhline(y=average_length, color='black', linestyle='--')
    plt.legend()

    # Add a color bar to show the density gradient
    norm = mcolors.Normalize(vmin=min_density, vmax=max_density)
    sm = plt.cm.ScalarMappable(cmap=colormap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=plt.gca(), orientation='vertical', pad=0.01)
    cbar.set_label('mA Density', rotation=270, labelpad=15)

    # Remove x-axis ticks entirely
    #plt.xticks([])
    plt.xticks(rotation=90)

    plt.title('Total CDR Lengths in Chromosome CDR')
    plt.xlabel('Chromosome')
    plt.ylabel('Total CDR Length')
    plt.tight_layout()
    output_path = os.path.join(output_dir, name + "_cdr_density_plot.png")
    #plt.savefig(output_path, format="png", dpi=dpi)
    plt.show()
    plt.close()

    print(f"Plot saved to {output_path}")

# Example usage:
plot_cdr_lengths_with_density_no_gap_mC(young_h3k9_match_cdr_regions, "young_H3K9me3", "/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot")
plot_cdr_lengths_with_density_no_gap_mC(old_h3k9_match_cdr_regions, "old_H3K9me3", "/private/groups/migalab/dan/data_analysis/HG002_figure3/bar_plot")




In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Helpers for lengths / keys
# -------------------------
def _ensure_length_col(df: pd.DataFrame) -> pd.Series:
    if "Region Length" in df.columns:
        return pd.to_numeric(df["Region Length"], errors="raise")
    # fall back to coordinates
    return (pd.to_numeric(df["CDR_end"], errors="raise")
            - pd.to_numeric(df["CDR_start"], errors="raise"))

def _extract_homolog(chrom: str):
    """
    Split 'chr10_PATERNAL' -> ('chr10', 'PATERNAL').
    Returns (base, homolog) or (None, None) if pattern doesn't match.
    """
    m = re.match(r"^(chr[0-9XYM]+)_(MATERNAL|PATERNAL)$", str(chrom), flags=re.IGNORECASE)
    if not m:
        return None, None
    base, hom = m.group(1), m.group(2).upper()
    return base, hom

def _resolve_ref_key(assembly: dict, chrom: str) -> str | None:
    """
    Try to find the correct sequence key in `assembly` for a given chromosome label.
    1) exact match,
    2) if label looks like 'chr10_MATERNAL'/'chr10_PATERNAL', fall back to base 'chr10'.
    """
    chrom = str(chrom)
    if chrom in assembly:
        return chrom
    m = re.match(r"^(chr[0-9XYM]+)_(MATERNAL|PATERNAL)$", chrom, flags=re.IGNORECASE)
    if m:
        base = m.group(1)
        if base in assembly:
            return base
    return None

def _count_As_in_row(row, assembly: dict,
                     chrom_col="Chromosome", start_col="CDR_start", end_col="CDR_end") -> int:
    """
    Count number of 'A' (case-insensitive) bases in the region [start, end) on the reference.
    Assumes 0-based, half-open coordinates.
    """
    if assembly is None:
        return np.nan
    ref_key = _resolve_ref_key(assembly, row[chrom_col])
    if ref_key is None:
        return np.nan
    seq = assembly[ref_key]
    start = int(row[start_col])
    end   = int(row[end_col])
    # Clamp to valid bounds
    start = max(0, min(start, len(seq)))
    end   = max(0, min(end,   len(seq)))
    if end <= start:
        return 0
    sub = seq[start:end]
    # Count both 'A' and 'a'
    return sub.count('A') + sub.count('a')

# ---------------------------------------
# Main: build per-chromosome homolog diffs
# ---------------------------------------
def build_homolog_diffs(df: pd.DataFrame,
                        density_col: str = "mA density",
                        drop_unpaired: bool = False,
                        *,
                        assembly: dict | None = None,
                        chrom_col: str = "Chromosome",
                        start_col: str = "CDR_start",
                        end_col: str = "CDR_end",
                        a_count_col: str = "A_count") -> pd.DataFrame:
    """
    Aggregate by homolog and compute per-chromosome differences:
      - size_diff_bp = total_length(MATERNAL) - total_length(PATERNAL)
      - mA_diff      = total_(density × #A)(MATERNAL) - total_(density × #A)(PATERNAL)

    A_count source:
      - If `a_count_col` exists in df, use it.
      - Else, if `assembly` is provided, compute A_count from coordinates.
      - Else, raise an error (we need #A to weight the density).
    """
    # Prepare columns
    length = _ensure_length_col(df)
    density = pd.to_numeric(df[density_col], errors="coerce").fillna(0.0)

    # Determine/compute #A per row
    if a_count_col in df.columns:
        a_count = pd.to_numeric(df[a_count_col], errors="coerce").fillna(0.0)
    else:
        if assembly is None:
            raise KeyError(
                f"'{a_count_col}' not found and no `assembly` provided to compute it. "
                "Pass an assembly dict or add an A_count column first."
            )
        a_count = df.apply(lambda r: _count_As_in_row(r, assembly,
                                                      chrom_col=chrom_col,
                                                      start_col=start_col,
                                                      end_col=end_col), axis=1).astype(float)
        a_count = a_count.fillna(0.0)

    # Map base chromosome & homolog
    base_hom = df[chrom_col].map(lambda c: _extract_homolog(c))
    base = base_hom.map(lambda t: t[0])
    hom  = base_hom.map(lambda t: t[1])

    # Build working table; NOTE: mA_amount now uses A_count, not length
    work = pd.DataFrame({
        "base_chrom": base,
        "homolog": hom,
        "length": length,                 # still used for size diff on x-axis
        "mA_amount": density * a_count    # <-- key change: density × (#A in region)
    }).dropna(subset=["base_chrom", "homolog"])

    # Sum per base chrom + homolog
    g = work.groupby(["base_chrom", "homolog"], as_index=False).sum(numeric_only=True)

    # Pivot to MATERNAL / PATERNAL columns
    length_piv = g.pivot(index="base_chrom", columns="homolog", values="length").fillna(0)
    ma_piv     = g.pivot(index="base_chrom", columns="homolog", values="mA_amount").fillna(0)

    # Optionally keep only chromosomes that have both homologs present
    if drop_unpaired:
        have_both = length_piv.columns.isin(["MATERNAL", "PATERNAL"]).all() and \
                    (length_piv.get("MATERNAL", 0) > 0) & (length_piv.get("PATERNAL", 0) > 0)
        length_piv = length_piv[have_both]
        ma_piv = ma_piv.loc[length_piv.index]

    # Ensure both columns exist
    for col in ["MATERNAL", "PATERNAL"]:
        if col not in length_piv.columns: length_piv[col] = 0
        if col not in ma_piv.columns:     ma_piv[col]     = 0

    # Final summary
    summary = pd.DataFrame({
        "Chromosome": length_piv.index,
        "size_mat_bp": length_piv["MATERNAL"].astype("int64"),
        "size_pat_bp": length_piv["PATERNAL"].astype("int64"),
        "mA_mat": ma_piv["MATERNAL"],
        "mA_pat": ma_piv["PATERNAL"],
    }).reset_index(drop=True)

    summary["size_diff_bp"] = summary["size_mat_bp"] - summary["size_pat_bp"]
    summary["mA_diff"]      = summary["mA_mat"]      - summary["mA_pat"]
    return summary

# -------------------------
# Scatter plotting (unchanged)
# -------------------------
def plot_size_vs_mA_diff(summary_df: pd.DataFrame, title: str,
                         annotate=False):
    x = summary_df["size_diff_bp"].to_numpy(dtype=float)
    y = summary_df["mA_diff"].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(5,5))
    ax.plot(x, y, marker='o', linestyle='None', alpha=0.8)

    
    # square panel box, flexible data aspect
    ax.set_aspect('auto')
    try:
        ax.set_box_aspect(1)
    except AttributeError:
        pass

    ax.set_xlabel("Size difference (Maternal − Paternal) [bp]")
    ax.set_ylabel("mA amount difference (Maternal − Paternal)  (density × #A)")
    ax.set_title(title)

    if annotate:
        for chrom, xi, yi in zip(summary_df["Chromosome"], x, y):
            ax.text(xi, yi, str(chrom), fontsize=8)

    fig.tight_layout()
    fig.savefig('/private/groups/migalab/dan/data_analysis/HG002_figure3/CENPA_mA_length_count.png', dpi=1600)
    return fig, ax


In [ ]:
assembly_ = open("/private/groups/migalab/dan/reference/hg002v1.0.1.fasta", "r")

start_time = time.time()

#Load the reference genome and make it into a dictionary 
fasta_sequences = SeqIO.parse(assembly_, "fasta")
assembly={}
for fasta in fasta_sequences:
    name, sequence = fasta.id, str(fasta.seq)
    assembly[name] = sequence

In [ ]:
summary = build_homolog_diffs(
    young_CENPA_match_cdr_regions,
    density_col="mA density",
    assembly=assembly,              # pass your reference dict
    chrom_col="Chromosome",
    start_col="CDR_start",
    end_col="CDR_end"
)
fig, ax = plot_size_vs_mA_diff(summary, title="Maternal − Paternal (A-weighted)")
